<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width="400px" style="opacity:0.7">
</center>

In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



# Chunking Pipeline
Here, we explore the chunking pipeline starting with chunking along section boundaries first, then breaking further into semantic chunks, and finally leveraging late interactions to perform "late chunking" to bring in any context that may have been lost during the semantic chunking process.

## Document ingestion → load document (PDF, Word, markdown).
Use Docling to parse & chunk into parent chunks (each section/chapter)

In [ ]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
import os
doc = DocumentConverter().convert(source=f"{os.getenv("BOOTCAMP_ROOT_DIR")}/data/ai_agents/ai_agents_1.pdf").document
chunker = HybridChunker()

#chunks = list(chunker.chunk(dl_doc=doc))
parent_chunks = list(chunker.chunk(doc))


2025-12-26 12:17:08,917 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-26 12:17:08,939 - INFO - Going to convert document batch...
2025-12-26 12:17:08,940 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2025-12-26 12:17:08,945 - INFO - Loading plugin 'docling_defaults'
2025-12-26 12:17:08,948 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-12-26 12:17:08,952 - INFO - Loading plugin 'docling_defaults'
2025-12-26 12:17:08,959 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-12-26 12:17:11,011 - INFO - Auto OCR model selected ocrmac.
2025-12-26 12:17:11,016 - INFO - Loading plugin 'docling_defaults'
2025-12-26 12:17:11,018 - INFO - Registered layout engines: ['docling_layout_default', 'docling_experimental_table_crops_layout']
2025-12-26 12:17:11,025 - INFO - Accelerator device: 'mps'
2025-12-26 12:18:07,896 - INFO - Loading plugin 'docling_defaul

## Semantic chunking 
Use `chonkie` to further semantically chunk each of above `parent` chunks

In [3]:
from chonkie import SemanticChunker

semantic_chunker = SemanticChunker() # Use defaults


2025-12-26 12:20:28,160 - INFO - No cached model found for minishlab/potion-base-32M, loading from local or hub.
2025-12-26 12:20:28,160 - INFO - Folder does not exist locally, attempting to use huggingface hub.


In [4]:
parent_chunk_id = 0
parent_chunk_dict_list = [] # A running list of all parent chunks (used during late chunking)

for p in parent_chunks:
    parent_chunk_dict = {}
    parent_chunk_dict["id"] = parent_chunk_id
    parent_chunk_dict["text"] = p.text
    semantic_chunk_dict_list = []
    parent_chunk_dict["semantic_chunks"] = semantic_chunk_dict_list
    parent_chunk_dict_list.append(parent_chunk_dict)
    sem_chunks = semantic_chunker.chunk(p.text)
    semantic_chunk_id = 0
    for sc in sem_chunks:
        # Maintain a list of semantic chunks for each parent chunk
        semantic_chunk_dict = {}
        semantic_chunk_dict["id"] = semantic_chunk_id
        semantic_chunk_dict["text"] = sc.text
        semantic_chunk_dict["start_char"] = sc.start_index
        semantic_chunk_dict["end_char"] = sc.end_index
        semantic_chunk_dict_list.append(semantic_chunk_dict)
        semantic_chunk_id += 1
    parent_chunk_id += 1

## Late Chunking
Use the `Late Chunking` approach along lines of how it is explained in [Jina AI](https://github.com/jina-ai/late-chunking)

In [7]:
from transformers import AutoTokenizer, AutoModel
import torch
import json

model_name = "jinaai/jina-embeddings-v2-base-en"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, output_hidden_states=True)

def late_chunk_parent(parent_chunk):
    """
    parent_chunk: dict with keys {id, text, semantic_chunks: List[{start_char, end_char, text, id}]}
    Returns same structure with added embeddings for each semantic chunk.
    """
    text = parent_chunk["text"]
    sem_chunks = parent_chunk["semantic_chunks"]

    # Tokenize + embed the *parent chunk text only*
    inputs = tokenizer(text, return_tensors="pt", truncation=False, return_offsets_mapping=True)
    # Save offsets separately
    offsets = inputs.pop("offset_mapping")[0].tolist()
    with torch.no_grad():
        outputs = model(**inputs)
    token_embs = outputs.last_hidden_state.squeeze(0)

    enriched_semantics = []
    for sc in sem_chunks:
        s, e = sc["start_char"], sc["end_char"]
        indices = [i for i, (ts, te) in enumerate(offsets) if te > s and ts < e]
        if not indices:
            continue
        emb = token_embs[indices].mean(dim=0).cpu().numpy()
        enriched_semantics.append({
            "semantic_id": sc["id"],
            "parent_id": parent_chunk["id"],
            "parent_text": parent_chunk["text"],
            "text": sc["text"],
            "num_tokens": len(indices),            
            "embedding": emb.tolist(),
        })
    return enriched_semantics

# Example loop over all parent chunks
all_embeddings = []
os.makedirs(f"{os.getenv("BOOTCAMP_ROOT_DIR")}/output", exist_ok=True)
with open(f"{os.getenv("BOOTCAMP_ROOT_DIR")}/output/late_chunks.jsonl", "w") as f:
    for parent in parent_chunk_dict_list:   # parent_chunks from Docling
        enriched_semantics = late_chunk_parent(parent)
        all_embeddings.extend(enriched_semantics)
        for es in enriched_semantics:
            f.write(json.dumps(es) + "\n")


## Sample Query embedding 
Thi will be used during a nearest neighbor search

In [8]:
query = "What is an AI agent?"
query_vector =model.encode(query)

Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
query_vector

array([-3.72436613e-01, -4.81530070e-01,  2.64331400e-01, -2.94817775e-01,
        2.95000702e-01,  7.32402086e-01,  3.97212237e-01, -4.13678527e-01,
        4.48365420e-01,  3.80905211e-01, -7.05723166e-01, -1.04914159e-01,
       -3.25901806e-01,  1.54613137e-01, -2.81944305e-01,  9.66732383e-01,
        1.93759397e-01,  2.49377742e-01, -2.91074038e-01, -2.58823425e-01,
       -3.70939076e-01, -1.37767047e-02, -1.32475972e+00, -7.79653251e-01,
        3.59717488e-01,  6.23237193e-01,  1.54673064e+00,  6.76619470e-01,
        6.63758218e-01,  5.64107776e-01,  1.01756012e+00, -4.84541059e-03,
       -1.26623422e-01,  3.79495770e-01, -3.60564262e-01, -7.76964307e-01,
       -5.68769813e-01,  4.40018535e-01,  1.70535892e-01,  2.70266324e-01,
        5.16504794e-03,  2.78177634e-02, -6.30791843e-01,  5.11587024e-01,
       -3.83724034e-01,  3.28658223e-01, -3.47152561e-01, -1.36503577e-01,
        3.65108609e-01, -1.26707003e-01, -4.13585752e-01, -2.89074063e-01,
        1.02632008e-01, -